<a href="https://colab.research.google.com/github/Zhalil24/BreastMRI-CNN-Classification/blob/main/DenseNet121NewResult.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# 1. KÜTÜPHANELER
# ============================================================

import os
import copy
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms, models

import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve,
    confusion_matrix
)

from google.colab import drive
from IPython.display import display


# ============================================================
# 2. DRIVE BAĞLAMA, DATASET AÇMA VE ÇIKTI KLASÖRLERİ
# ============================================================

import shutil

drive.mount('/content/drive')

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Kullanılan Cihaz: {device}")

# Dataset rar dosyasını Drive'dan Colab ortamına kopyala
rar_source_path = "/content/drive/MyDrive/data_set_zip/breast_mri_dataset.rar"
rar_target_path = "/content/breast_mri_dataset.rar"

shutil.copy(rar_source_path, rar_target_path)

# Dataseti /content içine aç
os.system(f'unrar x -o+ "{rar_target_path}" /content/')

# Dataset yolunu otomatik bul
def find_dataset_root(base_path="/content"):
    for root, dirs, files in os.walk(base_path):
        if "train" in dirs and "test" in dirs:
            return root
    raise FileNotFoundError("train ve test klasörlerini içeren dataset klasörü bulunamadı.")

data_dir = find_dataset_root("/content")

print("Bulunan dataset yolu:", data_dir)
print("Train klasörü:", os.path.join(data_dir, "train"))
print("Test klasörü:", os.path.join(data_dir, "test"))

# Çıktı klasörleri
output_root = '/content/drive/MyDrive/breast_mri_outputs'
charts_dir = os.path.join(output_root, 'charts')
tables_dir = os.path.join(output_root, 'tables')

os.makedirs(output_root, exist_ok=True)
os.makedirs(charts_dir, exist_ok=True)
os.makedirs(tables_dir, exist_ok=True)

print(f"Çıktı klasörü: {output_root}")


# ============================================================
# 3. VERİ DÖNÜŞÜMLERİ
# ============================================================

mri_mean = [0.485, 0.456, 0.406]
mri_std = [0.229, 0.224, 0.225]

train_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(10),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.3),
    transforms.RandomAffine(
        degrees=0,
        translate=(0.05, 0.05),
        scale=(0.95, 1.05)
    ),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.GaussianBlur(kernel_size=(3, 3), sigma=(0.1, 2.0)),
    transforms.ToTensor(),
    transforms.Normalize(mri_mean, mri_std)
])

val_test_transforms = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mri_mean, mri_std)
])


# ============================================================
# 4. VERİ SETLERİ
# ============================================================

def get_dataset(root, transform):
    return datasets.ImageFolder(os.path.join(data_dir, root), transform=transform)


# Eğitim için augmentation'lı train dataset
full_train_dataset = get_dataset('train', train_transforms)

# Fold validation için augmentation'sız train dataset
# Aynı train klasörü, ama validation tarafında rastgele dönüşüm olmaması için ayrı tanımlandı.
full_train_eval_dataset = get_dataset('train', val_test_transforms)

# Mevcut val ve test klasörleri korunuyor.
val_dataset = get_dataset('val', val_test_transforms)
test_dataset = get_dataset('test', val_test_transforms)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

class_names = full_train_dataset.classes
print("Sınıflar:", full_train_dataset.class_to_idx)

# Grafik etiketleri
if len(class_names) == 2:
    display_class_names = ['Benign', 'Malignant']
else:
    display_class_names = class_names


# ============================================================
# 5. MODEL OLUŞTURMA
# ============================================================

def build_model():
    model = models.densenet121(
        weights=models.DenseNet121_Weights.IMAGENET1K_V1
    )

    for param in model.parameters():
        param.requires_grad = False

    for param in model.features.denseblock2.parameters():
        param.requires_grad = True

    for param in model.features.denseblock3.parameters():
        param.requires_grad = True

    for param in model.features.denseblock4.parameters():
        param.requires_grad = True

    num_ftrs = model.classifier.in_features

    model.classifier = nn.Sequential(
        nn.Linear(num_ftrs, 2)
    )

    return model.to(device)


# ============================================================
# 6. OPTIMIZER VE LOSS
# ============================================================

def get_optimizer(model):
    optimizer = optim.Adam([
        {
            'params': model.features.denseblock2.parameters(),
            'lr': 1e-5
        },
        {
            'params': model.features.denseblock3.parameters(),
            'lr': 5e-5
        },
        {
            'params': model.features.denseblock4.parameters(),
            'lr': 1e-4
        },
        {
            'params': model.classifier.parameters(),
            'lr': 1e-3
        }
    ])

    return optimizer


criterion = nn.CrossEntropyLoss()


# ============================================================
# 7. EĞİTİM VE VALIDATION FONKSİYONLARI
# ============================================================

def train_one_epoch(model, loader, optimizer):
    model.train()

    running_loss = 0.0
    correct = 0

    for images, labels in loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc


def validate(model, loader):
    model.eval()

    running_loss = 0.0
    correct = 0

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = outputs.max(1)
            correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = correct / len(loader.dataset)

    return epoch_loss, epoch_acc


# ============================================================
# 8. DETAYLI DEĞERLENDİRME FONKSİYONU
# ============================================================

def evaluate_model(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            probs = torch.softmax(outputs, dim=1)[:, 1]
            _, predicted = outputs.max(1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    all_labels = np.array(all_labels)
    all_preds = np.array(all_preds)
    all_probs = np.array(all_probs)

    metrics = {
        'accuracy': accuracy_score(all_labels, all_preds),
        'precision': precision_score(all_labels, all_preds, zero_division=0),
        'recall': recall_score(all_labels, all_preds, zero_division=0),
        'f1_score': f1_score(all_labels, all_preds, zero_division=0)
    }

    try:
        metrics['roc_auc'] = roc_auc_score(all_labels, all_probs)
    except ValueError:
        metrics['roc_auc'] = np.nan

    return metrics, all_labels, all_preds, all_probs


# ============================================================
# 9. GRAFİK KAYDETME FONKSİYONLARI
# ============================================================

def plot_accuracy(history, fold_no, save_path):
    epochs_range = range(1, len(history['train_acc']) + 1)

    plt.figure(figsize=(9, 6))
    plt.plot(
        epochs_range,
        history['train_acc'],
        marker='o',
        label='Train Accuracy'
    )
    plt.plot(
        epochs_range,
        history['val_acc'],
        marker='o',
        label='Validation Accuracy'
    )

    plt.title(f'Fold {fold_no}/5 Accuracy Grafiği')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


def plot_loss(history, fold_no, save_path):
    epochs_range = range(1, len(history['train_loss']) + 1)

    plt.figure(figsize=(9, 6))
    plt.plot(
        epochs_range,
        history['train_loss'],
        marker='o',
        label='Train Loss'
    )
    plt.plot(
        epochs_range,
        history['val_loss'],
        marker='o',
        label='Validation Loss'
    )

    plt.title(f'Fold {fold_no}/5 Loss Grafiği')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


def plot_confusion_matrix_graph(labels, preds, title, save_path):
    cm = confusion_matrix(labels, preds, labels=[0, 1])

    plt.figure(figsize=(7, 6))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=display_class_names,
        yticklabels=display_class_names
    )

    plt.title(title)
    plt.ylabel('Gerçek Sınıf')
    plt.xlabel('Tahmin Edilen Sınıf')
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


def plot_roc_curve_graph(labels, probs, title, save_path):
    plt.figure(figsize=(8, 6))

    try:
        fpr, tpr, _ = roc_curve(labels, probs)
        auc_value = roc_auc_score(labels, probs)

        plt.plot(
            fpr,
            tpr,
            lw=2,
            label=f'ROC Curve - AUC = {auc_value:.4f}'
        )
        plt.plot(
            [0, 1],
            [0, 1],
            linestyle='--',
            lw=2,
            label='Random Classifier'
        )

    except ValueError:
        plt.text(
            0.5,
            0.5,
            'ROC hesaplanamadı.\nValidation setinde tek sınıf olabilir.',
            ha='center',
            va='center'
        )

    plt.title(title)
    plt.xlabel('False Positive Rate')
    plt.ylabel('True Positive Rate')
    plt.legend(loc='lower right')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()
    plt.close()


# ============================================================
# 10. 5-FOLD STRATIFIED CROSS VALIDATION
# ============================================================

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_indices = np.arange(len(full_train_dataset))
train_labels = np.array(full_train_dataset.targets)

fold_results = []
fold_summary_rows = []
epoch_history_rows = []
output_file_rows = []

best_overall_val_acc = -1
best_model = None
first_fold_model = None

# İstersen eski kod gibi ilk fold modelini test etmek için bunu "first_fold" yapabilirsin.
# Tavsiye edilen: "best_fold"
FINAL_MODEL_POLICY = "best_fold"

epochs = 10
batch_size = 32

for fold_no, (train_idx, val_idx) in enumerate(
    skf.split(train_indices, train_labels),
    start=1
):
    print(f"\n--- FOLD {fold_no}/5 ---")

    train_sub = Subset(full_train_dataset, train_idx)
    val_sub = Subset(full_train_eval_dataset, val_idx)

    train_loader = DataLoader(
        train_sub,
        batch_size=batch_size,
        shuffle=True,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    val_loader = DataLoader(
        val_sub,
        batch_size=batch_size,
        shuffle=False,
        num_workers=2,
        pin_memory=torch.cuda.is_available()
    )

    model = build_model()
    optimizer = get_optimizer(model)

    history = {
        'train_loss': [],
        'train_acc': [],
        'val_loss': [],
        'val_acc': []
    }

    best_epoch_val_acc = -1
    best_epoch_no = 1
    best_epoch_state = None

    for epoch in range(epochs):
        t_loss, t_acc = train_one_epoch(model, train_loader, optimizer)
        v_loss, v_acc = validate(model, val_loader)

        history['train_loss'].append(t_loss)
        history['train_acc'].append(t_acc)
        history['val_loss'].append(v_loss)
        history['val_acc'].append(v_acc)

        epoch_history_rows.append({
            'Fold': fold_no,
            'Epoch': epoch + 1,
            'Train_Loss': t_loss,
            'Train_Accuracy': t_acc,
            'Validation_Loss': v_loss,
            'Validation_Accuracy': v_acc
        })

        print(
            f"Epoch {epoch+1}: "
            f"T_Loss: {t_loss:.4f} | "
            f"T_Acc: {t_acc:.4f} | "
            f"V_Loss: {v_loss:.4f} | "
            f"V_Acc: {v_acc:.4f}"
        )

        if v_acc > best_epoch_val_acc:
            best_epoch_val_acc = v_acc
            best_epoch_no = epoch + 1
            best_epoch_state = copy.deepcopy(model.state_dict())

    # Fold içindeki en iyi epoch modeli geri yükleniyor.
    if best_epoch_state is not None:
        model.load_state_dict(best_epoch_state)

    fold_results.append(history)

    # Fold validation değerlendirmesi
    val_metrics, val_labels, val_preds, val_probs = evaluate_model(model, val_loader)

    print(f"\nFold {fold_no} Validation Performansı")
    print(f"Best Epoch: {best_epoch_no}")
    print(f"Accuracy:  {val_metrics['accuracy']:.4f}")
    print(f"Precision: {val_metrics['precision']:.4f}")
    print(f"Recall:    {val_metrics['recall']:.4f}")
    print(f"F1-Score:  {val_metrics['f1_score']:.4f}")
    print(f"ROC-AUC:   {val_metrics['roc_auc']:.4f}")

    # Her fold için grafik dosya yolları
    accuracy_path = os.path.join(charts_dir, f'fold_{fold_no}_accuracy.png')
    loss_path = os.path.join(charts_dir, f'fold_{fold_no}_loss.png')
    cm_path = os.path.join(charts_dir, f'fold_{fold_no}_confusion_matrix.png')
    roc_path = os.path.join(charts_dir, f'fold_{fold_no}_roc_curve.png')

    # Her fold için ayrı grafikler
    plot_accuracy(history, fold_no, accuracy_path)
    plot_loss(history, fold_no, loss_path)

    plot_confusion_matrix_graph(
        val_labels,
        val_preds,
        title=f'Fold {fold_no}/5 Validation Confusion Matrix',
        save_path=cm_path
    )

    plot_roc_curve_graph(
        val_labels,
        val_probs,
        title=f'Fold {fold_no}/5 Validation ROC-AUC Grafiği',
        save_path=roc_path
    )

    fold_summary_rows.append({
        'Fold': fold_no,
        'Best_Epoch': best_epoch_no,
        'Validation_Accuracy': val_metrics['accuracy'],
        'Validation_Precision': val_metrics['precision'],
        'Validation_Recall': val_metrics['recall'],
        'Validation_F1_Score': val_metrics['f1_score'],
        'Validation_ROC_AUC': val_metrics['roc_auc'],
        'Accuracy_Graph': accuracy_path,
        'Loss_Graph': loss_path,
        'Confusion_Matrix_Graph': cm_path,
        'ROC_Curve_Graph': roc_path
    })

    output_file_rows.extend([
        {
            'Output_Type': 'Fold Accuracy Graph',
            'Fold': fold_no,
            'File_Path': accuracy_path
        },
        {
            'Output_Type': 'Fold Loss Graph',
            'Fold': fold_no,
            'File_Path': loss_path
        },
        {
            'Output_Type': 'Fold Confusion Matrix Graph',
            'Fold': fold_no,
            'File_Path': cm_path
        },
        {
            'Output_Type': 'Fold ROC Curve Graph',
            'Fold': fold_no,
            'File_Path': roc_path
        }
    ])

    # Eski koddaki gibi Fold 1 modeli de saklanıyor.
    if fold_no == 1:
        first_fold_model = copy.deepcopy(model)

    # Final test için en iyi fold modeli saklanıyor.
    if val_metrics['accuracy'] > best_overall_val_acc:
        best_overall_val_acc = val_metrics['accuracy']
        best_model = copy.deepcopy(model)


# ============================================================
# 11. FOLD TABLOLARI
# ============================================================

fold_summary_df = pd.DataFrame(fold_summary_rows)
epoch_history_df = pd.DataFrame(epoch_history_rows)

print("\n--- FOLD ÖZET TABLOSU ---")
display(fold_summary_df)

print("\n--- EPOCH GEÇMİŞ TABLOSU ---")
display(epoch_history_df)


# ============================================================
# 12. FINAL MODEL SEÇİMİ VE TEST DEĞERLENDİRMESİ
# ============================================================

if FINAL_MODEL_POLICY == "first_fold":
    final_model = first_fold_model
    final_model_name = "Fold 1 Model"
else:
    final_model = best_model
    final_model_name = "Best Fold Model"

test_metrics, test_labels, test_preds, test_probs = evaluate_model(
    final_model,
    test_loader
)

print("\n--- TEST SETİ PERFORMANSI ---")
print(f"Final Model: {final_model_name}")
print(f"Accuracy:  {test_metrics['accuracy']:.4f}")
print(f"Precision: {test_metrics['precision']:.4f}")
print(f"Recall:    {test_metrics['recall']:.4f}")
print(f"F1-Score:  {test_metrics['f1_score']:.4f}")
print(f"ROC-AUC:   {test_metrics['roc_auc']:.4f}")

final_test_df = pd.DataFrame([{
    'Final_Model': final_model_name,
    'Test_Accuracy': test_metrics['accuracy'],
    'Test_Precision': test_metrics['precision'],
    'Test_Recall': test_metrics['recall'],
    'Test_F1_Score': test_metrics['f1_score'],
    'Test_ROC_AUC': test_metrics['roc_auc']
}])

print("\n--- FINAL TEST TABLOSU ---")
display(final_test_df)


# ============================================================
# 13. MEVCUT ORTALAMA ACCURACY GRAFİĞİ
# ============================================================
# Bu grafik eski kodda vardı, korunuyor.

all_train_acc = np.array([
    history['train_acc']
    for history in fold_results
])

all_val_acc = np.array([
    history['val_acc']
    for history in fold_results
])

mean_train_acc = np.mean(all_train_acc, axis=0)
mean_val_acc = np.mean(all_val_acc, axis=0)

epochs_range = range(1, len(mean_train_acc) + 1)

average_accuracy_path = os.path.join(
    charts_dir,
    'average_training_vs_validation_accuracy.png'
)

plt.figure(figsize=(9, 6))
plt.plot(
    epochs_range,
    mean_train_acc,
    marker='o',
    label='Average Train Acc'
)
plt.plot(
    epochs_range,
    mean_val_acc,
    marker='o',
    label='Average Val Acc'
)

plt.title('Training vs Validation Accuracy - 5 Fold Ortalama')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(average_accuracy_path, dpi=300, bbox_inches='tight')
plt.show()
plt.close()

output_file_rows.append({
    'Output_Type': 'Average Accuracy Graph',
    'Fold': 'All',
    'File_Path': average_accuracy_path
})


# ============================================================
# 14. MEVCUT TEST CONFUSION MATRIX GRAFİĞİ
# ============================================================
# Bu grafik eski kodda vardı, korunuyor.

final_test_cm_path = os.path.join(
    charts_dir,
    'final_test_confusion_matrix.png'
)

plot_confusion_matrix_graph(
    test_labels,
    test_preds,
    title='Final Test Confusion Matrix',
    save_path=final_test_cm_path
)

output_file_rows.append({
    'Output_Type': 'Final Test Confusion Matrix Graph',
    'Fold': 'Final Test',
    'File_Path': final_test_cm_path
})


# ============================================================
# 15. MEVCUT TEST ROC CURVE GRAFİĞİ
# ============================================================
# Bu grafik eski kodda vardı, korunuyor.

final_test_roc_path = os.path.join(
    charts_dir,
    'final_test_roc_curve.png'
)

plot_roc_curve_graph(
    test_labels,
    test_probs,
    title='Final Test ROC Curve',
    save_path=final_test_roc_path
)

output_file_rows.append({
    'Output_Type': 'Final Test ROC Curve Graph',
    'Fold': 'Final Test',
    'File_Path': final_test_roc_path
})


# ============================================================
# 16. TÜM SONUÇLARI EXCEL DOSYASINA KAYDETME
# ============================================================

output_files_df = pd.DataFrame(output_file_rows)

excel_path = os.path.join(
    tables_dir,
    'breast_mri_model_results.xlsx'
)

with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
    fold_summary_df.to_excel(
        writer,
        sheet_name='Fold_Summary',
        index=False
    )

    epoch_history_df.to_excel(
        writer,
        sheet_name='Epoch_History',
        index=False
    )

    final_test_df.to_excel(
        writer,
        sheet_name='Final_Test',
        index=False
    )

    output_files_df.to_excel(
        writer,
        sheet_name='Output_Files',
        index=False
    )

print("\n--- KAYIT İŞLEMİ TAMAMLANDI ---")
print(f"Grafikler klasörü: {charts_dir}")
print(f"Excel dosyası: {excel_path}")

print("\nÜretilen çıktı dosyaları:")
display(output_files_df)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Kullanılan Cihaz: cuda
Çıktı klasörü: /content/drive/MyDrive/breast_mri_outputs


FileNotFoundError: [Errno 2] No such file or directory: '/content/breast_mri_dataset/train'